In [1]:
import pandas as pd
import numpy as np
import os

# Đường dẫn tới các file CSV trong thư mục của bạn
file_paths = ['2019-Oct.csv', '2019-Nov.csv']

print("Đang tải và gộp dữ liệu... (Quá trình này có thể mất chút thời gian nếu file lớn)")
df_list = [pd.read_csv(file) for file in file_paths]
df = pd.concat(df_list, ignore_index=True)

print(f"Kích thước dữ liệu ban đầu: {df.shape}")
df.head() # Xem thử 5 dòng đầu

Đang tải và gộp dữ liệu... (Quá trình này có thể mất chút thời gian nếu file lớn)


MemoryError: Unable to allocate 2.51 GiB for an array with shape (5, 67501979) and data type object

In [ ]:
print("Đang kiểm tra dữ liệu trùng lặp...")
duplicates = df.duplicated().sum()

if duplicates > 0:
    df = df.drop_duplicates(keep='first')
    print(f"Đã xóa {duplicates} dòng trùng lặp.")
else:
    print("Không có dòng trùng lặp.")
    
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đang kiểm tra dữ liệu trùng lặp...
Đã xóa 130739 dòng trùng lặp.
Kích thước dữ liệu hiện tại: (109820004, 9)


In [ ]:
# Sửa 'event_time' thành tên cột thời gian thực tế trong file của bạn
time_col = 'event_time' 

if time_col in df.columns:
    print(f"Đang chuyển đổi cột '{time_col}' sang Datetime...")
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    print("Hoàn tất chuyển đổi.")
else:
    print(f"Không tìm thấy cột '{time_col}'. Các cột hiện có: {df.columns.tolist()}")

Đang chuyển đổi cột 'event_time' sang Datetime...
Hoàn tất chuyển đổi.


In [ ]:
# Kiểm tra tỷ lệ missing
missing_percent = (df.isnull().sum() / len(df)) * 100
print("Tỷ lệ dữ liệu thiếu (%):\n", missing_percent[missing_percent > 0])

# Xử lý missing cho cột dạng chuỗi (Categorical)
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# Xử lý missing cho cột dạng số (Numeric) bằng giá trị trung vị (Median)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print("\nĐã xử lý xong dữ liệu thiếu.")

Tỷ lệ dữ liệu thiếu (%):
 category_code    32.219708
brand            13.960305
user_session      0.000011
dtype: float64

Đã xử lý xong dữ liệu thiếu.


In [ ]:
# Thay 'price' bằng cột giá trị thực tế bạn muốn kiểm tra
target_col = 'price'

if target_col in df.columns:
    outlier_count = len(df[df[target_col] < 0])
    if outlier_count > 0:
        df = df[df[target_col] >= 0]
        print(f"Đã xóa {outlier_count} dòng có giá trị '{target_col}' bị âm.")
    else:
        print(f"Cột '{target_col}' không có giá trị âm.")

Cột 'price' không có giá trị âm.


In [ ]:
# Tạo thư mục 'cleaned' nếu nó chưa có sẵn
os.makedirs('cleaned', exist_ok=True)

output_path = 'cleaned/cleaned_data.csv'
print(f"Đang lưu dữ liệu ra file {output_path}...")

# Export ra file csv (không lưu cột index)
df.to_csv(output_path, index=False)

print("Lưu file thành công!")
print(f"Kích thước bộ dữ liệu cuối cùng: {df.shape}")

Đang lưu dữ liệu ra file cleaned/cleaned_data.csv...
Lưu file thành công!
Kích thước bộ dữ liệu cuối cùng: (109820004, 9)
